# HERMES Boundary Lab — Colab runner

> **Toy holography-inspired neural-memory experiment; not a physical black-hole simulation.**
>
> No black-hole physics, no information paradox, no AdS/CFT. "Bulk" and "boundary" are
> borrowed names for a bottleneck: a 10x10 binary grid is encoded into a ring of slots, the
> grid is withheld, and a decoder must answer `(row, col)` queries from the ring alone.

Use this notebook for anything heavy: long training runs, big rings, and sweeps.
The Streamlit app is for interactive inspection on CPU.

**Before running:** `Runtime -> Change runtime type -> GPU` (a T4 is plenty).

## 1. Environment

In [ ]:
!nvidia-smi || echo "No GPU attached — training will fall back to CPU."

In [ ]:
REPO_URL = "https://github.com/rk0604/HERMES.git"
BRANCH = "main"

import os
import sys

if not os.path.exists("HERMES"):
    !git clone --branch $BRANCH $REPO_URL
else:
    !cd HERMES && git pull --ff-only

%cd HERMES
sys.path.insert(0, os.getcwd())

# torch and numpy already ship with Colab; install the rest quietly.
!pip install -q matplotlib streamlit pytest

In [ ]:
import json

import numpy as np
import torch

from hermes import DISCLAIMER, __version__
from hermes.analysis import (
    boundary_subset_curve,
    bulk_to_boundary_influence,
    default_subset_sizes,
    evaluate_dataset,
    reconstruct_grid,
)
from hermes.config import DataConfig, ExperimentConfig, ModelConfig, TrainConfig
from hermes.data import generate_dataset, generate_sample
from hermes.interventions import AblationSpec, compare_ablation, slot_mask_from_indices
from hermes.training import resolve_device, save_checkpoint, save_run_record, train_model
from hermes.visualization import (
    plot_boundary_bar,
    plot_boundary_ring,
    plot_reconstruction_panels,
    plot_subset_curves,
    plot_training_history,
)

print(f"HERMES v{__version__}")
print(f"torch {torch.__version__}, device {resolve_device('auto')}")
print(DISCLAIMER)

In [ ]:
# Fast sanity check that the clone is healthy (~15s).
!python -m pytest -q

## 2. Train

A GPU makes it cheap to train a much bigger run than the CPU demo. Note the learning-rate
warmup: early in training the encoder's attention is near-uniform, so the boundary barely
varies with the bulk and the model sits at the all-zeros baseline. The warmup plus cosine
decay shortens that stall considerably.

In [ ]:
config = ExperimentConfig(
    data=DataConfig(train_size=16384, val_size=2048, seed=0),
    model=ModelConfig(
        hidden_dim=128,
        boundary_dim=64,
        num_boundary_slots=20,
        num_heads=8,
    ),
    train=TrainConfig(
        epochs=60,
        lr=4e-3,
        batch_size=256,
        queries_per_grid=100,
        seed=0,
        device="auto",
    ),
    name="colab-baseline",
    notes="GPU run: wider model, 16k training grids.",
)

result = train_model(config, verbose=True)
print(f"\nTrained in {result.duration_seconds:.1f}s ({result.model.num_parameters():,} params)")

In [ ]:
metrics = result.final_metrics
print(f"cell accuracy      : {metrics['cell_accuracy']:.4f}")
print(f"exact grid match   : {metrics['exact_grid_match_rate']:.4f}   <- headline metric")
print(f"BCE                : {metrics['bce']:.4f}")
print(f"all-zeros baseline : {metrics['majority_baseline_accuracy']:.4f}   <- compare against this")
print("\nper pattern:")
for pattern, value in sorted(metrics["accuracy_by_pattern"].items()):
    exact = metrics["exact_match_by_pattern"][pattern]
    print(f"  {pattern:<18} cell {value:.4f}   exact {exact:.4f}")

plot_training_history(result.history)

In [ ]:
model = result.model
checkpoint = save_checkpoint(
    f"checkpoints/{config.name}.pt", model, config, result.history, result.final_metrics
)
record = save_run_record(f"runs/{config.name}/run.json", config, result.final_metrics)
print(checkpoint, record, sep="\n")

## 3. Reconstruct a single bulk

All 100 cells are queried independently; the decoder sees only the boundary.

In [ ]:
sample = generate_sample(pattern="circle", seed=7)
recon = reconstruct_grid(model, sample.grid)

print(f"{sample.describe()}")
print(f"cell accuracy {recon.accuracy:.4f}, errors {int(recon.error_map.sum())}, "
      f"exact match {recon.exact_match}")
plot_reconstruction_panels(
    recon.truth, recon.probabilities, recon.prediction, recon.error_map
)

## 4. Which slots matter? (causal)

Ablate each slot on its own and measure the accuracy drop across a fresh evaluation set.
Unlike attention weights, this is causal evidence that a slot is being used.

In [ ]:
eval_set = generate_dataset(2048, DataConfig(seed=4242))
baseline = evaluate_dataset(model, eval_set)
num_slots = model.num_boundary_slots

drops = []
for slot in range(num_slots):
    mask = slot_mask_from_indices([slot], num_slots)
    ablated = evaluate_dataset(model, eval_set, slot_mask=mask)
    drops.append(baseline["cell_accuracy"] - ablated["cell_accuracy"])
drops = np.array(drops)

print(f"baseline {baseline['cell_accuracy']:.4f}")
print(f"mean single-slot drop {drops.mean():.4f}, max {drops.max():.4f} (slot {drops.argmax()})")
plot_boundary_bar(drops, title="Accuracy drop from ablating each slot alone", ylabel="drop")

In [ ]:
plot_boundary_ring(drops, title="Per-slot causal importance")

In [ ]:
# Structured vs. random removal at a matched budget: does *where* on the ring matter?
budget = num_slots // 2
rng = np.random.default_rng(0)

def masked_accuracy(slots):
    mask = slot_mask_from_indices(slots, num_slots)
    return evaluate_dataset(model, eval_set, slot_mask=mask)["cell_accuracy"]

every_other = masked_accuracy(AblationSpec(strategy="every_other").resolve(num_slots))
random_runs = [
    masked_accuracy(AblationSpec(strategy="random", count=budget, seed=int(s)).resolve(num_slots))
    for s in rng.integers(0, 10_000, size=8)
]
contiguous_runs = [
    masked_accuracy(
        AblationSpec(strategy="contiguous", start=int(s), length=budget).resolve(num_slots)
    )
    for s in range(0, num_slots, max(1, num_slots // 8))
]

print(f"removing {budget}/{num_slots} slots (baseline {baseline['cell_accuracy']:.4f}):")
print(f"  every other : {every_other:.4f}")
print(f"  random      : {np.mean(random_runs):.4f} +/- {np.std(random_runs):.4f}")
print(f"  contiguous  : {np.mean(contiguous_runs):.4f} +/- {np.std(contiguous_runs):.4f}")
print("\nA large contiguous-vs-random gap would say the encoding is spatially organised")
print("on the ring rather than uniformly distributed.")

## 5. Boundary-subset decoding

How much of the ring do you actually need? Slots outside the subset are hidden from the
decoder's cross-attention.

In [ ]:
sizes = default_subset_sizes(num_slots)
contiguous = boundary_subset_curve(model, eval_set.grids, sizes=sizes, mode="contiguous", trials=8)
random_curve = boundary_subset_curve(model, eval_set.grids, sizes=sizes, mode="random", trials=8)

for i, size in enumerate(sizes):
    print(f"{size:>3} slots   contiguous {contiguous.accuracy_mean[i]:.4f}   "
          f"random {random_curve.accuracy_mean[i]:.4f}")

plot_subset_curves(
    [contiguous, random_curve],
    baseline=baseline["cell_accuracy"],
    majority_baseline=baseline["majority_baseline_accuracy"],
)

## 6. Sweep: does ring size change the picture?

This is the kind of run that justifies a GPU. Each ring size is a full training run, and
every config, seed and metric is written to `runs/ring-sweep.json`.

The narrow rings are the interesting ones: at 20 slots x 64 dims the boundary is nowhere near
an information bottleneck, so the model is not forced to be clever. Shrinking the ring until
accuracy actually degrades is what makes the layout question meaningful.

In [ ]:
sweep = []
for slots in [4, 8, 12, 20, 32]:
    sweep_config = config.replace(
        **{
            "model.num_boundary_slots": slots,
            "train.epochs": 40,
            "data.train_size": 8192,
            "name": f"ring-{slots}",
        }
    )
    run = train_model(sweep_config, verbose=False)
    entry = {
        "num_boundary_slots": slots,
        "boundary_floats": sweep_config.model.boundary_floats,
        "num_parameters": run.model.num_parameters(),
        "cell_accuracy": run.final_metrics["cell_accuracy"],
        "exact_grid_match_rate": run.final_metrics["exact_grid_match_rate"],
        "bce": run.final_metrics["bce"],
        "majority_baseline": run.final_metrics["majority_baseline_accuracy"],
        "config": sweep_config.to_dict(),
    }
    sweep.append(entry)
    print(f"{slots:>3} slots  cell {entry['cell_accuracy']:.4f}  "
          f"exact {entry['exact_grid_match_rate']:.4f}  "
          f"(baseline {entry['majority_baseline']:.4f})")

os.makedirs("runs", exist_ok=True)
with open("runs/ring-sweep.json", "w", encoding="utf-8") as handle:
    json.dump({"disclaimer": DISCLAIMER, "sweep": sweep}, handle, indent=2)
print("\nSaved runs/ring-sweep.json")

## 7. Keep the results

Colab discards the filesystem when the runtime recycles. Either download the checkpoint or
copy it to Drive.

In [ ]:
# Option A: download to your machine.
from google.colab import files

files.download(f"checkpoints/{config.name}.pt")

In [ ]:
# Option B: copy checkpoints and run records to Drive.
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/HERMES"
!cp -r checkpoints runs "/content/drive/MyDrive/HERMES/"
!ls -la "/content/drive/MyDrive/HERMES"

---

### Reading the numbers honestly

- These grids are sparse, so predicting all zeros already scores ~0.86 cell accuracy. Always
  compare against `majority_baseline_accuracy`; prefer `exact_grid_match_rate`.
- Attention maps (`bulk_to_boundary_influence`) are association measures. Ablation and subset
  decoding are the causal tests.
- At the default width the boundary is not a real information bottleneck. Until the ring is
  narrow enough (or quantised/noisy enough) that capacity binds, this is a query-conditioned
  autoencoder — see "What would distinguish HERMES from an ordinary autoencoder" in the README.

**Toy holography-inspired neural-memory experiment; not a physical black-hole simulation.**